# SMS Spam Detection with Machine Learning

This notebook trains a high-accuracy text classification model using **TF-IDF Vectorization** and **Multinomial Naive Bayes** to detect whether an SMS is **Ham** (legitimate) or **Spam**.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Load and Inspect Dataset
The dataset is tab-separated with no header. We use `quoting=3` (QUOTE_NONE) to handle messages containing quotation marks.

In [ ]:
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'], quoting=3)
print(f"Total messages: {len(df)}")
print("\nClass counts:")
print(df['label'].value_counts())
df.head()

## 2. Train / Test Split
We perform an 80/20 train-test split with stratification to preserve class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['message'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)
print(f"Training set: {len(X_train)} messages")
print(f"Test set:     {len(X_test)} messages")

## 3. Build & Train Pipeline
We use a `Pipeline` combining:
- `TfidfVectorizer`: Unigrams and bigrams, lowercased, English stop words removed, sublinear term-frequency scaling.
- `MultinomialNB`: Laplace smoothing tuned to `alpha=0.1` for optimal spam detection.

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        lowercase=True,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('classifier', MultinomialNB(alpha=0.1))
])

model.fit(X_train, y_train)
print("Model training complete!")

## 4. Evaluation on Test Set

In [ ]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Confusion Matrix:")
print(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=['ham', 'spam']),
                   index=['Actual Ham', 'Actual Spam'],
                   columns=['Predicted Ham', 'Predicted Spam']))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

## 5. Save the Model

In [ ]:
joblib.dump(model, 'spam_detector_model.joblib')
print("Model saved to spam_detector_model.joblib")

## 6. Test with Custom Messages

In [ ]:
def check_sms(text):
    pred = model.predict([text])[0]
    probs = model.predict_proba([text])[0]
    classes = list(model.classes_)
    confidence = max(probs) * 100
    print(f"Message: '{text}'")
    print(f"Result : [{pred.upper()}] with {confidence:.2f}% confidence\n")

# Try your own messages here:
check_sms("Hey mate, are we still meeting at 6?")
check_sms("CONGRATS! You have won £1,000 cash. Reply WIN to claim now!")